# Evaluasi Model Adaptive LoRa — (MedGemma Chest X-Ray)

## Prepare GPU

In [ ]:
"""
AUTO-SELECT GPU
===============
Memindai semua GPU yang tersedia via nvidia-smi, lalu memilih satu GPU
dengan VRAM bebas terbanyak dan utilisasi terendah.
CUDA_VISIBLE_DEVICES di-set SEBELUM torch di-import agar efektif.
"""

import os
import subprocess
import sys

N_GPUS_TO_USE = 1   # ← ubah sesuai kebutuhan

QUERY_FIELDS = "index,name,memory.free,memory.total,utilization.gpu"
try:
    result = subprocess.run(
        ["nvidia-smi", f"--query-gpu={QUERY_FIELDS}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    )
except FileNotFoundError:
    sys.exit("ERROR: nvidia-smi tidak ditemukan. Pastikan driver NVIDIA terinstall.")
except subprocess.CalledProcessError as e:
    sys.exit(f"ERROR: nvidia-smi gagal: {e.stderr.strip()}")

gpu_info = []
for line in result.stdout.strip().splitlines():
    parts = [p.strip() for p in line.split(",")]
    if len(parts) < 5:
        continue
    try:
        gpu_info.append({
            "index"    : int(parts[0]),
            "name"     : parts[1],
            "mem_free" : int(parts[2]),
            "mem_total": int(parts[3]),
            "util_gpu" : int(parts[4]),
        })
    except ValueError:
        continue

if not gpu_info:
    sys.exit("ERROR: Tidak ada GPU yang terdeteksi dari nvidia-smi.")

header = f"{'IDX':>3}  {'Name':<30}  {'Free MiB':>10}  {'Total MiB':>10}  {'Util%':>6}"
sep    = "=" * len(header)
print(sep)
print("  GPU INFO (nvidia-smi)")
print(sep)
print(f"  {header}")
print("-" * len(header))
for g in gpu_info:
    marker = " ◄" if g is sorted(gpu_info, key=lambda x: (-x["mem_free"], x["util_gpu"]))[0] else ""
    print(
        f"  {g['index']:>3}  {g['name']:<30}  "
        f"{g['mem_free']:>10,}  {g['mem_total']:>10,}  {g['util_gpu']:>5}%{marker}"
    )
print(sep)

sorted_gpus  = sorted(gpu_info, key=lambda g: (-g["mem_free"], g["util_gpu"]))
selected     = sorted_gpus[:N_GPUS_TO_USE]
selected_ids = [str(g["index"]) for g in selected]

# *** WAJIB di-set SEBELUM import torch ***
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected_ids)

print(f"\nGPU terpilih (N_GPUS_TO_USE={N_GPUS_TO_USE}):")
for rank, g in enumerate(selected):
    print(
        f"  cuda:{rank} ← GPU {g['index']}  {g['name']}"
        f"  |  VRAM bebas: {g['mem_free']:,} MiB / {g['mem_total']:,} MiB"
        f"  |  Util: {g['util_gpu']}%"
    )
print(f"\nCUDA_VISIBLE_DEVICES = \"{os.environ['CUDA_VISIBLE_DEVICES']}\"")
print("OK — lanjutkan ke cell berikutnya.")

## 1. Konfigurasi & Persiapan Data

Konfigurasi path, dekripsi dataset, dan validasi adapter.

In [ ]:
import io
import getpass
import torch
import pandas as pd
from pathlib import Path
from cryptography.fernet import Fernet, InvalidToken

# ============================================================
# KONFIGURASI PATH — sesuaikan dengan lokasi file Anda
# ============================================================
ENCRYPTED_CSV_PATH  = Path("SIAP_TRAINING.csv.encrypted")
ENCRYPTED_IMAGE_DIR = Path("./IMAGE/GAMBAR_TERBARU_ENKRIPSI")

# Path adapter hasil training AdaLoRA v3
ADAPTER_DIR = "./LORA_ADAPTER_MEDGEMMA_4B_IT"

# Random seed untuk reproducibility split
RANDOM_STATE = 42

# ============================================================
# INPUT ENCRYPTION KEY (Fernet) — dimasukkan saat runtime
# ============================================================
raw_key = getpass.getpass("Masukkan Fernet encryption key (input tersembunyi): ").strip()
if not raw_key:
    raise ValueError("Encryption key tidak boleh kosong.")

try:
    fernet = Fernet(raw_key.encode() if isinstance(raw_key, str) else raw_key)
    fernet.decrypt(fernet.encrypt(b"key_validation_test"))
    print("Encryption key valid.")
except InvalidToken:
    raise ValueError("Encryption key tidak valid — pastikan format Fernet yang benar.")
except Exception as e:
    raise ValueError(f"Error validasi key: {e}")

# ============================================================
# VALIDASI PATH
# ============================================================
assert ENCRYPTED_CSV_PATH.exists(),  f"CSV tidak ditemukan: {ENCRYPTED_CSV_PATH}"
assert ENCRYPTED_IMAGE_DIR.exists(), f"Folder gambar tidak ditemukan: {ENCRYPTED_IMAGE_DIR}"
assert Path(ADAPTER_DIR).exists(),   f"Adapter tidak ditemukan: {ADAPTER_DIR}"

# ============================================================
# DEKRIPSI CSV DI MEMORI
# ============================================================
with open(ENCRYPTED_CSV_PATH, "rb") as f:
    encrypted_csv_bytes = f.read()

try:
    decrypted_csv_bytes = fernet.decrypt(encrypted_csv_bytes)
except InvalidToken:
    raise ValueError("Gagal dekripsi CSV — periksa encryption key.")

df = pd.read_csv(io.BytesIO(decrypted_csv_bytes))

REQUIRED_COLS = ["image_file", "findings", "conclusion", "exam_type"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"Kolom CSV tidak lengkap, kurang: {missing}"

df = df.dropna(subset=REQUIRED_COLS)
df["findings"]   = df["findings"].astype(str)
df["conclusion"] = df["conclusion"].astype(str)
df["exam_type"]  = df["exam_type"].astype(str)
df = df.reset_index(drop=True)

print(f"Total data  : {len(df)} baris")
print(f"Adapter dir : {ADAPTER_DIR}")
print(f"\nDistribusi exam_type:\n{df['exam_type'].value_counts()}")


## 2. Reproduksi Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["exam_type"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=RANDOM_STATE, stratify=temp_df["exam_type"]
)

print(f"Train : {len(train_df)} sampel")
print(f"Val   : {len(val_df)} sampel")
print(f"Test  : {len(test_df)} sampel  ← digunakan untuk evaluasi")
print(f"\nDistribusi Test:\n{test_df['exam_type'].value_counts()}")

# Reset index agar iterasi bersih
test_df = test_df.reset_index(drop=True)


## 3. Load Model + Adaptive LoRA Adapter

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import PeftConfig, PeftModel

# ============================================================
# BACA KONFIGURASI ADAPTER UNTUK MENDAPAT BASE MODEL ID
# ============================================================
print(f"Membaca konfigurasi adapter dari: {ADAPTER_DIR}")
peft_config    = PeftConfig.from_pretrained(ADAPTER_DIR)
BASE_MODEL_ID  = peft_config.base_model_name_or_path
print(f"Base model    : {BASE_MODEL_ID}")

# ============================================================
# LOAD PROCESSOR
# ============================================================
# Gunakan processor yang disimpan bersama adapter agar
# konfigurasi tokenizer dan image processor identik dengan saat training
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
print("Processor loaded dari adapter dir.")

# ============================================================
# LOAD BASE MODEL
# ============================================================
n_gpu      = torch.cuda.device_count()
max_memory = {i: "170GiB" for i in range(n_gpu)}
max_memory["cpu"] = "64GB"

print(f"\nMemuat base model {BASE_MODEL_ID} dalam bfloat16...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
)

# ============================================================
# TEMPELKAN ADALORA ADAPTER KE BASE MODEL
# ============================================================
print(f"Menempelkan AdaLoRA adapter dari {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# Ringkasan memori setelah load
print()
for i in range(n_gpu):
    alloc  = torch.cuda.memory_allocated(i) / 1e9
    total  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {alloc:.1f} / {total:.1f} GB")

print("\nModel + AdaLoRA Adapter siap untuk evaluasi.")

## 4. Fungsi Helper

In [ ]:
import io
from PIL import Image
from cryptography.fernet import Fernet, InvalidToken

# ============================================================
# FUNGSI HELPER — IDENTIK DENGAN SAAT TRAINING
# ============================================================

SYSTEM_PROMPT = [
    "Anda adalah dokter radiologi. Buat laporan thorax lengkap dalam bahasa Indonesia.",
    "Tolong analisis X-ray dada ini dan tuliskan temuan radiologis beserta kesimpulannya.",
    "Sebagai spesialis radiologi, apa diagnosis Anda untuk gambar rontgen berikut?",
    "Berikan evaluasi klinis dari citra radiografi thorax ini secara terperinci.",
    "Tuliskan laporan medis dari foto rontgen dada ini, mencakup temuan dan konklusi."
]

def build_inference_prompt(processor) -> str:
    """Format inferensi: hanya sisi user, tanpa jawaban model."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": SYSTEM_PROMPT},
            ],
        },
    ]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def build_reference_text(exam_type: str, findings: str, conclusion: str) -> str:
    """
    Susun teks referensi (ground truth) dengan format yang SAMA
    dengan output yang diharapkan dari model saat training.
    """
    return (
        f"Exam Type:\n{exam_type.strip()}\n\n"
        f"Findings:\n{findings.strip()}\n\n"
        f"Conclusion:\n{conclusion.strip()}"
    )


def decrypt_image_to_pil(fernet: Fernet, img_path: Path) -> Image.Image:
    """Dekripsi file .enc di memori → PIL Image. Tidak menyentuh disk."""
    with open(img_path, "rb") as f:
        encrypted_bytes = f.read()
    try:
        decrypted_bytes = fernet.decrypt(encrypted_bytes)
    except InvalidToken as e:
        raise RuntimeError(f"Gagal dekripsi {img_path.name}: {e}")
    img = Image.open(io.BytesIO(decrypted_bytes)).convert("RGB")
    img = img.resize((896, 896), Image.Resampling.LANCZOS)
    return img


print("Helper functions siap.")
print(f"Jumlah data test : {len(test_df)} sampel")
print(f"\nContoh referensi :")
ex = test_df.iloc[0]
print(build_reference_text(ex["exam_type"], ex["findings"], ex["conclusion"])[:300] + "...")

## 5. Inferensi pada Test Set

In [ ]:
import torch
from tqdm.auto import tqdm

# ============================================================
# INFERENSI MODEL PADA SELURUH DATA TEST
# ============================================================
# Model menghasilkan prediksi teks untuk setiap gambar X-ray
# di test set, kemudian dibandingkan dengan ground truth
# menggunakan metrik ROUGE, BERTScore, dan BLEU.
# ============================================================

semua_prediksi  = []   # hasil generate model
semua_referensi = []   # ground truth laporan dokter

prompt_text = build_inference_prompt(processor)

print(f"Memulai inferensi pada {len(test_df)} data test...")
print("(Setiap sampel: dekripsi gambar → generate teks → decode output)\n")

model.eval()
with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferensi"):

        # 1. Dekripsi dan siapkan gambar
        img_path = ENCRYPTED_IMAGE_DIR / row["image_file"]
        image    = decrypt_image_to_pil(fernet, img_path)

        # 2. Tokenisasi prompt + gambar
        inputs   = processor(
            images=image,
            text=prompt_text,
            return_tensors="pt",
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        # 3. Generate prediksi
        #    do_sample=False → greedy decoding (deterministik & reproducible)
        #    max_new_tokens=512 → cukup untuk laporan thorax lengkap
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,    # tidak berpengaruh saat do_sample=False
        )

        # 4. Decode hanya token yang di-generate (potong bagian prompt)
        generated_tokens = outputs[0][input_len:]
        prediksi_teks    = processor.decode(generated_tokens, skip_special_tokens=True).strip()

        # 5. Susun teks referensi (ground truth) sesuai format training
        referensi_teks = build_reference_text(
            row["exam_type"], row["findings"], row["conclusion"]
        )

        semua_prediksi.append(prediksi_teks)
        semua_referensi.append(referensi_teks)

print(f"\nInferensi selesai — {len(semua_prediksi)} prediksi dihasilkan.")
print("\n" + "="*65)
print("CONTOH PREDIKSI vs REFERENSI (sampel pertama)")
print("="*65)
print(f"\n[PREDIKSI MODEL]\n{semua_prediksi[0][:400]}...")
print(f"\n[REFERENSI DOKTER]\n{semua_referensi[0][:400]}...")

In [ ]:
import pandas as pd

# ============================================================
# Setelah proses inferensi selesai
# ============================================================

# Membuat DataFrame dengan hasil prediksi dan referensi
data = {
    'prediction': semua_prediksi,
    'ground_truth': semua_referensi
}

df_prediksi_referensi = pd.DataFrame(data)

# Menyimpan DataFrame ke dalam file CSV
output_csv_path = 'FINAL_ADALORA.csv'
df_prediksi_referensi.to_csv(output_csv_path, index=False)

print(f"Hasil inferensi telah disimpan ke {output_csv_path}")

## 6. Evaluasi Metrik

### ROUGE

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

tot_r1_prec = tot_r1_rec = tot_r1_f1 = 0.0
tot_r2_f1   = 0.0
tot_rl_f1   = 0.0
exact_match = 0
n_data      = len(semua_prediksi)

for pred, ref in zip(semua_prediksi, semua_referensi):
    pred_bersih = pred.strip().lower()
    ref_bersih  = ref.strip().lower()

    if pred_bersih == ref_bersih:
        exact_match += 1

    scores = scorer.score(ref_bersih, pred_bersih)

    tot_r1_prec += scores["rouge1"].precision
    tot_r1_rec  += scores["rouge1"].recall
    tot_r1_f1   += scores["rouge1"].fmeasure
    tot_r2_f1   += scores["rouge2"].fmeasure
    tot_rl_f1   += scores["rougeL"].fmeasure

avg_r1_prec = (tot_r1_prec / n_data) * 100
avg_r1_rec  = (tot_r1_rec  / n_data) * 100
avg_r1_f1   = (tot_r1_f1   / n_data) * 100
avg_r2_f1   = (tot_r2_f1   / n_data) * 100
avg_rl_f1   = (tot_rl_f1   / n_data) * 100
akurasi_em  = (exact_match  / n_data) * 100

print("\n" + "="*65)
print(" 📊 HASIL EVALUASI ROUGE — AdaLoRA v3")
print("="*65)
print(f"Jumlah data test          : {n_data} sampel")
print("-"*65)
print("ROUGE-1 (Kecocokan Kosakata / N-Gram Tunggal):")
print(f"  Precision               : {avg_r1_prec:.2f}%")
print(f"  Recall                  : {avg_r1_rec:.2f}%")
print(f"  F1-Score                : {avg_r1_f1:.2f}%")
print("-"*65)
print(f"ROUGE-2 F1 (Frasa 2 kata) : {avg_r2_f1:.2f}%")
print(f"ROUGE-L F1 (Alur kalimat) : {avg_rl_f1:.2f}%")
print("="*65)

# Simpan hasil untuk ringkasan akhir
rouge_results = {
    "r1_precision": avg_r1_prec,
    "r1_recall"   : avg_r1_rec,
    "r1_f1"       : avg_r1_f1,
    "r2_f1"       : avg_r2_f1,
    "rl_f1"       : avg_rl_f1,
    "exact_match" : akurasi_em,
}

### BERTScore

In [ ]:
import numpy as np
from bert_score import score as bert_score_fn

print("Menghitung BERTScore...")
print("(Memuat model bert-base-multilingual-cased ke memori...)\n")

P, R, F1 = bert_score_fn(
    semua_prediksi,
    semua_referensi,
    lang="id",
    verbose=True,
)

mean_p   = np.mean(P.numpy())  * 100
mean_r   = np.mean(R.numpy())  * 100
mean_f1  = np.mean(F1.numpy()) * 100

print("\n" + "="*65)
print(" 🧠 HASIL EVALUASI BERTSCORE — AdaLoRA v3")
print("="*65)
print("BERTScore (Kecocokan Makna / Semantik Medis):")
print(f"  Precision               : {mean_p:.2f}%")
print(f"  Recall                  : {mean_r:.2f}%")
print(f"  F1-Score                : {mean_f1:.2f}%")
print("="*65)

# Simpan hasil untuk ringkasan akhir
bertscore_results = {
    "precision": mean_p,
    "recall"   : mean_r,
    "f1"       : mean_f1,
}

## 7. Ringkasan & Simpan Hasil

In [ ]:
import json
from datetime import datetime

# ============================================================
# RINGKASAN AKHIR SELURUH METRIK
# ============================================================

print("\n" + "="*65)
print(" 🏆 RINGKASAN EVALUASI LENGKAP — AdaLoRA v3")
print("="*65)
print(f"  Adapter      : {ADAPTER_DIR}")
print(f"  Data test    : {len(test_df)} sampel")
print(f"  Waktu        : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*65)

print("\n📊 ROUGE:")
print(f"  ROUGE-1 Precision : {rouge_results['r1_precision']:.2f}%")
print(f"  ROUGE-1 Recall    : {rouge_results['r1_recall']:.2f}%")
print(f"  ROUGE-1 F1        : {rouge_results['r1_f1']:.2f}%")
print(f"  ROUGE-2 F1        : {rouge_results['r2_f1']:.2f}%")
print(f"  ROUGE-L F1        : {rouge_results['rl_f1']:.2f}%")

print("\n🧠 BERTScore:")
print(f"  Precision         : {bertscore_results['precision']:.2f}%")
print(f"  Recall            : {bertscore_results['recall']:.2f}%")
print(f"  F1                : {bertscore_results['f1']:.2f}%")

print("\n📝 BLEU:")
print(f"  BLEU-1            : {bleu_results['bleu_1']:.2f}%")
print(f"  BLEU-2            : {bleu_results['bleu_2']:.2f}%")
print(f"  BLEU-3            : {bleu_results['bleu_3']:.2f}%")
print(f"  BLEU-4            : {bleu_results['bleu_4']:.2f}%")

print("="*65)

# ============================================================
# SIMPAN HASIL KE FILE JSON
# ============================================================
hasil_evaluasi = {
    "adapter_dir"  : ADAPTER_DIR,
    "n_test"       : len(test_df),
    "timestamp"    : datetime.now().isoformat(),
    "rouge"        : rouge_results,
    "bertscore"    : bertscore_results,
    "bleu"         : bleu_results,
}

output_json = Path(ADAPTER_DIR) / "eval_results.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(hasil_evaluasi, f, ensure_ascii=False, indent=2)

print(f"\n✅ Hasil evaluasi disimpan di: {output_json}")


### RadGraph

#### Translate to English with MarianT

In [ ]:
from radgraph import F1RadGraph

print("Memuat model F1RadGraph...")
scorer_radgraph = F1RadGraph(
    reward_level="partial",   # partial = token partial match
    model_type="radgraph-xl", # radgraph-xl lebih akurat
    device="cpu"              # ganti ke "cuda:0" jika GPU tersedia
)

print("Menghitung RadGraph F1 untuk baseline...\n")
outputs_rg = scorer_radgraph(hyps=semua_prediksi, refs=semua_referensi)

mean_rg_score = outputs_rg[0]   # rata-rata F1 seluruh laporan
all_rg_scores = outputs_rg[1]   # skor per laporan

# Tambahkan skor per baris ke DataFrame
result_df["radgraph_f1"] = all_rg_scores

print("\n" + "="*65)
print(" 🏥 HASIL EVALUASI RADGRAPH F1 — BASELINE (Tanpa Adapter)")
print("="*65)
print(f"  Rata-rata RadGraph F1  : {mean_rg_score:.4f}")
print(f"  Std                    : {result_df['radgraph_f1'].std():.4f}")
print(f"  Min                    : {result_df['radgraph_f1'].min():.4f}")
print(f"  Max                    : {result_df['radgraph_f1'].max():.4f}")
print("="*65)

print("\nRata-rata RadGraph F1 per exam_type:")
print(result_df.groupby("exam_type")["radgraph_f1"].agg(["mean", "min", "max"]).round(4).to_string())

radgraph_results_baseline = {
    "mean": mean_rg_score,
    "std" : result_df["radgraph_f1"].std(),
    "min" : result_df["radgraph_f1"].min(),
    "max" : result_df["radgraph_f1"].max(),
}

#### Evaluation

In [ ]:
import pandas as pd
from radgraph import F1RadGraph

print("Membaca file CSV...")
df_eval = pd.read_csv("data.csv")

# Tarik ketiga kolom dari CSV
image_files = df_eval["image_file"].tolist() # <--- Tarik nama gambarnya
hypothesis = df_eval["prediction_en"].tolist()
ground_truth = df_eval["ground_truth_en"].tolist()

print("Sedang memuat model F1RadGraph...")
scorer = F1RadGraph(
    reward_level="partial", 
    model_type="radgraph-xl", 
    device="cpu" 
)

print("--- Memulai Evaluasi ---")
outputs = scorer(hyps=hypothesis, refs=ground_truth)

mean_score = outputs[0]
details = outputs[2]

print("\n" + "="*50)
print(f"⭐ Rata-rata RadGraph F1 Score (100 Laporan): {mean_score:.4f}")
print("="*50)

# ==========================================
# MENAMPILKAN CONTOH HASIL (MISAL: LAPORAN KE-1 / INDEX 0)
# ==========================================
indeks_contoh = 0  # Ganti angka ini (0-99) untuk melihat laporan gambar lainnya

print(f"\n🔍 DETAIL EVALUASI UNTUK GAMBAR: {image_files[indeks_contoh]}")
print(f"Ground Truth : {ground_truth[indeks_contoh]}")
print(f"Prediksi AI  : {hypothesis[indeks_contoh]}")
print("-" * 50)
print("Entitas Medis yang Ditemukan oleh RadGraph pada Prediksi AI:")

try:
    if isinstance(details, list):
        hyp_report = details[indeks_contoh] 
    elif isinstance(details, dict):
        hyp_report = details.get("hyp", [{}])[indeks_contoh]
    else:
        hyp_report = None

    if hyp_report and "entities" in hyp_report:
        for entity_id, entity_info in hyp_report["entities"].items():
            tokens = entity_info.get('tokens', 'N/A')
            label = entity_info.get('label', 'N/A')
            print(f"- {tokens} [{label}]")
    else:
        print("Tidak ada entitas yang ditemukan.")

except Exception as e:
    print(f"Gagal memproses detail: {e}")